#### В современной разработке (особенно в Computer Vision и NLP) архитектуру делят на две части:   
- **Backbone (тело/магистраль)** - это тяжелая часть сети (например, слои ResNet, ViT или блоки Transformer). Её задача — принять сырые данные (пиксели, токены текста) и превратить их в эмбеддинг (embedding) — вектор фичей (размерностью, например, 512 или 768), который компактно описывает смысл входных данных.

- **Head (голова)** - легковесная насадка (обычно 1-3 линейных слоя) на конце сети.  
Ее задача принять вектор фичей от бэкбона и форматировать его в конечный нужный нам API-ответ

#### Архитектура сети: от "входа" к "выходу"

В Нейросети информация течет строго в одном направлении:  

[Входные данные] ->

`Первые слои` (Нижние / Входные) -> Ищут базовые элементы: линии, углы, точки, градиенты.  

`Средние слои` -> Объединяют линии в простые фигуры: круги, квадраты, текстуры.  

`Верхние слои` (Последние / Выходные) -> Собирают сложные объекты: лица, машины, собак, дефекты.

-> [Результат]

>Первые слои (входные) ближе всего к сырым данным (картинке или тексту). Они универсальны, например распознование края обьекта, их не нужно переучивать.

>Верхение слои находятся в самом конце цепочки. Принимают финальное решение, их переделывают под конкретную задачу.

---

**Transfer Learning (Трансферное обучение)** - метод машинного обучения, при котором берется готовая предобученная модель (бэкбон) и адаптируется под новые цели.  
- Экономия данных - модель уже качественная
- Скорость обучения - быстрый процесс сходимости 
- Снижение затрат - требуется меньше GPU/TPU

Основные стратегии адаптации:

`Вариант 1: Заморозка` (Обучаем только «хвост»)

Мы намертво блокируем (замораживаем) веса всех первых и средних слоев. Они работают просто как «глаза», которые выдают набор базовых признаков. Мы пристыковываем в самый конец один новый выходной слой и обучаем **только его**.

- _Плюсы:_ Ультра-быстро, не ломает базовые знания сети, нужно мало данных.

`Вариант 2: Тонкая настройка / Fine-Tuning` (Дообучаем верхнюю половину или всё)

Это более глубокая адаптация. Мы берем и разрешаем менять веса не только у самого последнего слоя, но и у средних или даже у первых слоев.

- **Дообучение верхней половины**: Мы говорим сети: _«Твои входные слои идеальны, не трогай их. А вот средние и верхние слои давай немного подкорректируем, чтобы они лучше понимали специфику наших новых картинок»_.
- **Дообучение всей сети**: Мы разрешаем менять веса вообще во всех слоях, включая самые первые (входные). Но делаем скорость обучения (learning rate) микроскопической. Сеть аккуратно, едва заметно сдвигает свои настройки на всех уровнях под ваш датасет.

>Выбор стратегии зависит от размера датасета и схожести задачи в обученной модели

`Золотое правило`:  

|Размер вашего датасета|Новая задача ПОХОЖА на старую (например, была ImageNet, стали кожные заболевания)|Новая задача НЕ ПОХОЖА на старую (например, была ImageNet, стали спутниковые снимки Земли)|
|---|---|---|
|**Мало данных** (сотни примеров)|**Вариант 1 (Заморозка)**. Базовые признаки те же, данных на обучение мало — переобучим сеть, если разморозим.|**Тупик**. Размораживать нельзя (мало данных), замораживать бессмысленно (признаки не подходят). Нужно искать другой бэкбон.|
|**Много данных** (тысячи/десятки тысяч)|**Вариант 2 (Разморозка верха)**. Можно слегка подстроить верхние слои для идеальной точности.|**Вариант 2 (Разморозка всей сети)**. Мощный датасет позволит полностью перестроить модель под новые реалии.|

---

Полная заморозка (Feature Extraction)  
- Задача: Сортировка деталей на конвейере (всего 3 типа деталей)  

- Условия: Имеется по 100 фотографии каждой детали. Задачи похожи геометрией: на ImageNet были железные предметы, ложки, механизмы, так что базовые слои ResNet отлично видят металл, грани и формы  

- Решение: Замораживаем все, кроме финального слоя классификации

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

Сферы применения
- Компьютерное зрение **CV**: (ResNet, EfficientNet, YOLO, ImageNet) адаптируют для детекции дефектов на производстве или анализа медицинских снимков.   

- Обработка естественного языка **NLP**: (BERT, GPT, Llama) дообучают под юридические задачи, медицинские чат-боты, анализаторы отзывов.